# SALT3 Colab Runbook

This notebook is an execution checklist for the SALT3 workflow. It is intentionally lightweight: the real code lives in `salt3_common.py`, and model state lives in Google Drive under `/content/drive/MyDrive/SALT3`.

## Runtime model

Each Colab notebook starts from a clean runtime. It can import `.py` code from Drive, but it cannot see variables from another notebook. Durable dependencies must be saved as files under `/content/drive/MyDrive/SALT3`.

Use this pattern in every notebook: mount Drive, add the SALT3 code folder to `sys.path`, load config/artifacts from Drive, then write this notebook's outputs back to Drive.

In [1]:
from pathlib import Path
import shutil
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not running in Colab or Drive is already unavailable:', exc)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
CODE_DIR = PROJECT_ROOT / 'code'
CODE_DIR.mkdir(parents=True, exist_ok=True)

# If salt3_common.py is uploaded beside this notebook, copy it into Drive code/.
local_common = Path('/content/salt3_common.py')
if local_common.exists():
    shutil.copy2(local_common, CODE_DIR / 'salt3_common.py')

sys.path.insert(0, str(CODE_DIR))
sys.path.insert(0, '/content')
print('SALT3 project root:', PROJECT_ROOT)
print('SALT3 code dir    :', CODE_DIR)

Mounted at /content/drive
SALT3 project root: /content/drive/MyDrive/SALT3
SALT3 code dir    : /content/drive/MyDrive/SALT3/code


In [3]:
!pip install xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.0 MB/s eta 0:00:00


In [5]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    import os
    if os.environ.get('HF_TOKEN'):
        login(token=os.environ['HF_TOKEN'])

In [4]:
!python /content/drive/MyDrive/SALT3/scripts/localize_salt3_failure.py \
    /content/drive/MyDrive/SALT3/init/videberta_salt_init_v3_proj/model \
    --source chandar-lab/NeoBERT \
    --data_cache /content/drive/MyDrive/SALT3/datasets/culturax_vi_3000000_seq1024_tok3487ef147c62 \
    --n_chunks 16

Device: cuda

 LOAD MODELS
SALT init: /content/drive/MyDrive/SALT3/init/videberta_salt_init_v3_proj/model
Vocab: 30522  |  random-baseline loss = log(V) = 10.326
rotary.py: 100% 2.58k/2.58k [00:00<00:00, 7.55MB/s]
[transformers] A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
tokenizer_config.json: 100% 1.31k/1.31k [00:00<00:00, 4.21MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 3.83MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 11.6MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 445kB/s]


In [8]:
!python /content/drive/MyDrive/SALT3/scripts/test_decoder_global_map_and_freq_bias.py \
    /content/drive/MyDrive/SALT3/init/videberta_salt_init_v3_proj/model \
    --source chandar-lab/NeoBERT \
    --data_cache /content/drive/MyDrive/SALT3/datasets/culturax_vi_3000000_seq1024_tok3487ef147c62 \
    --n_eval 16 --n_count 8000

Loading dataset from disk: 100% 23/23 [00:00<00:00, 61.80it/s]
Counting Vietnamese unigram freq over 8000 train chunks...
Loading dataset from disk: 100% 23/23 [00:00<00:00, 58.68it/s]
  tokens counted: 8,192,000  |  bias range [-15.92, -3.28]

  variant                             loss
  freq-only (bias, no encoder)          7.286
  global map + VI-freq bias            10.182
  log(V) nominal floor                 10.326
  global map                           10.709
  projected + VI-freq bias             14.361
  projected current                    14.952

  alpha sweep: logits = alpha*(h @ W_global) + VI-freq bias
   alpha  loss
    0.00    7.286
    0.05    7.285
    0.10    7.308
    0.20    7.418
    0.30    7.607
    0.50    8.164
    0.75    9.092
    1.00   10.182

Best alpha = 0.05 -> loss 7.285
freq-only floor = 7.286
=> Encoder roughly neutral when down-weighted; small alpha + VI-freq bias is the safe init.


## Folder contract

The expected Drive layout is:

```text
/content/drive/MyDrive/SALT3/
  code/salt3_common.py
  init/<INIT_NAME>/model/
  runs/<RUN_NAME>/run_config.json
  runs/<RUN_NAME>/checkpoints/
  runs/<RUN_NAME>/final_model/
  runs/<RUN_NAME>/metrics.jsonl
  eval/<EVAL_NAME>/
  datasets/
```

A run is resumed only from its own `runs/<RUN_NAME>/checkpoints/`. A continuation run starts from another model artifact but writes into a new run folder.

In [ ]:
for child in ['code', 'init', 'runs', 'eval', 'datasets']:
    (PROJECT_ROOT / child).mkdir(parents=True, exist_ok=True)
print('Ready folders:')
for child in sorted(PROJECT_ROOT.iterdir()):
    print(' -', child)

Ready folders:
 - /content/drive/MyDrive/SALT3/code
 - /content/drive/MyDrive/SALT3/datasets
 - /content/drive/MyDrive/SALT3/eval
 - /content/drive/MyDrive/SALT3/init
 - /content/drive/MyDrive/SALT3/runs


## Run modes

Use `02_train_cpt_run.ipynb` with one of these modes:

```python
MODE = 'new'
RUN_NAME = 'simple_cpt_400k_lr1e-4'
BASE_MODEL_REF = 'init/videberta_salt_init_v1/model'

MODE = 'resume'
RUN_NAME = 'simple_cpt_400k_lr1e-4'

MODE = 'continue'
RUN_NAME = 'simple_cpt_2M_continue_lr5e-5'
BASE_MODEL_REF = 'runs/simple_cpt_2M/final_model'
```